In [17]:
!pip install memory_profiler pandas

In [18]:
!git clone https://github.com/PacktPublishing/Supercharged-Coding-with-Gen-AI.git

Cloning into 'Supercharged-Coding-with-Gen-AI'...
remote: Enumerating objects: 209, done.
remote: Counting objects: 100% (209/209), done.
remote: Compressing objects: 100% (156/156), done.
remote: Total 209 (delta 67), reused 178 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (209/209), 56.72 KiB | 2.27 MiB/s, done.
Resolving deltas: 100% (67/67), done.


In [19]:
%cd Supercharged-Coding-with-Gen-AI/ch14

/content/Supercharged-Coding-with-Gen-AI/ch14/Supercharged-Coding-with-Gen-AI/ch14/Supercharged-Coding-with-Gen-AI/ch14/Supercharged-Coding-with-Gen-AI/ch14


In [20]:
!pip install memory_profiler pandas
import pandas as pd
import numpy as np
import os

def create_mock_csv(filename, rows, cols):
    chunk_size = 1000
    with open(filename, 'w') as f:
        headers = [f"video_{i}" for i in range(cols)]
        f.write("," + ",".join(headers) + "\n")
        for i in range(0, rows, chunk_size):
            current_chunk = min(chunk_size, rows - i)
            data = np.random.rand(current_chunk, cols)
            df = pd.DataFrame(data, index=[f"user_{j}" for j in range(i, i + current_chunk)])
            df.to_csv(f, header=False)

create_mock_csv('interactions_100.csv', 100, 100)
create_mock_csv('interactions_1000.csv', 1000, 1000)
create_mock_csv('interactions_10_000.csv', 10000, 10000)
print("Đã chuẩn bị xong môi trường và dữ liệu!")

Đã chuẩn bị xong môi trường và dữ liệu!


In [21]:
import time

def fibonacci_recursive(n):
    if n <= 1: return n
    return fibonacci_recursive(n - 1) + fibonacci_recursive(n - 2)

print("Bắt đầu đo thời gian chạy hàm Fibonacci đệ quy:")
for n in range(10, 42, 5):
    start_time = time.time()
    fibonacci_recursive(n)
    end_time = time.time()
    print(f"Runtime for fibonacci_recursive({n}): {end_time - start_time:.4f} seconds")

Bắt đầu đo thời gian chạy hàm Fibonacci đệ quy:
Runtime for fibonacci_recursive(10): 0.0000 seconds
Runtime for fibonacci_recursive(15): 0.0002 seconds
Runtime for fibonacci_recursive(20): 0.0035 seconds
Runtime for fibonacci_recursive(25): 0.0311 seconds
Runtime for fibonacci_recursive(30): 0.3779 seconds
Runtime for fibonacci_recursive(35): 5.2766 seconds
Runtime for fibonacci_recursive(40): 25.9247 seconds


In [24]:
%%writefile profile_space.py
import pandas as pd
from memory_profiler import profile

@profile
def get_top_video(path):
    # Thêm index_col=0 để bỏ qua cột chứa chữ 'user_...' khi tính toán
    interactions = pd.read_csv(path, index_col=0)
    avg_ratio = interactions.mean(axis=0, skipna=True)
    return avg_ratio.idxmax()

if __name__ == '__main__':
    paths = ['interactions_100.csv', 'interactions_1000.csv', 'interactions_10_000.csv']
    for p in paths:
        print(f"\n--- Đang xử lý file: {p} ---")
        print("Top video:", get_top_video(p))

Overwriting profile_space.py


In [25]:
!python profile_space.py


--- Đang xử lý file: interactions_100.csv ---
Filename: /content/Supercharged-Coding-with-Gen-AI/ch14/Supercharged-Coding-with-Gen-AI/ch14/Supercharged-Coding-with-Gen-AI/ch14/Supercharged-Coding-with-Gen-AI/ch14/profile_space.py

Line #    Mem usage    Increment  Occurrences   Line Contents
     4    123.6 MiB    123.6 MiB           1   @profile
     5                                         def get_top_video(path):
     6                                             # Thêm index_col=0 để bỏ qua cột chứa chữ 'user_...' khi tính toán
     7    125.4 MiB      1.9 MiB           1       interactions = pd.read_csv(path, index_col=0)
     8    125.5 MiB      0.1 MiB           1       avg_ratio = interactions.mean(axis=0, skipna=True)
     9    125.6 MiB      0.1 MiB           1       return avg_ratio.idxmax()


Top video: video_28

--- Đang xử lý file: interactions_1000.csv ---
Filename: /content/Supercharged-Coding-with-Gen-AI/ch14/Supercharged-Coding-with-Gen-AI/ch14/Supercharged-Coding-w

In [26]:
import pandas as pd
import time

def get_top_video_chunking(path):
    cumulative_sum = None
    cumulative_count = None
    chunksize = 1000

    print(f"Đang đọc file {path} theo từng cục (chunk) {chunksize} dòng...")
    # Thêm index_col=0 để tránh lỗi chữ như lúc nãy
    for chunk in pd.read_csv(path, chunksize=chunksize, index_col=0):
        chunk_sum = chunk.sum(skipna=True)
        chunk_count = chunk.count()

        if cumulative_sum is None:
            cumulative_sum = chunk_sum
            cumulative_count = chunk_count
        else:
            cumulative_sum += chunk_sum
            cumulative_count += chunk_count

    average_ratio = cumulative_sum / cumulative_count
    return average_ratio.idxmax()

# Bắt đầu đo thời gian chạy hàm đã tối ưu
start_csv = time.time()
top_vid = get_top_video_chunking('interactions_10_000.csv')
end_csv = time.time()

print(f"✅ Kết quả: {top_vid}")
print(f"⏳ Thời gian hoàn thành: {end_csv - start_csv:.4f} giây")
print("Thuật toán tối ưu đã chạy thành công mà không làm tràn bộ nhớ!")

Đang đọc file interactions_10_000.csv theo từng cục (chunk) 1000 dòng...
✅ Kết quả: video_3024
⏳ Thời gian hoàn thành: 46.3331 giây
Thuật toán tối ưu đã chạy thành công mà không làm tràn bộ nhớ!
